# Binder structures colored by pLDDT

For each of the 3 **binders** (`query == 1`) we visualise the predicted TCR:pMHC structure colored by per-residue **pLDDT**.

For each binder's **peptide**, we also show the structure with the **highest pMHC:TCR PAE** among all targets carrying that same peptide (the worst-docking model for that peptide), again colored by pLDDT — as a contrast to the confident binder.

Structures + pLDDT arrays come from the TCRdock run in `TCRdock/` (PDBs relocated to `TCRdock/analysis/`). Binder / peptide labels come from the previous-run CSV, matched by TCR+pMHC **sequence** (targetids were renumbered in this run).

## 1. Imports and paths

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import py3Dmol

BASE = Path("TCRdock")
PDB_DIR = Path("structures")           # relocated model PDBs
NPY_DIR = Path("plddt")                        # plddt/pae .npy live in TCRdock/
RESULTS_TSV = Path("data/cancer_output_w_pae.tsv")   # this run (renumbered targetids)
OLD_CSV = Path("data/combined_df_merged_stats_pyrosetta_previous_runApr2025.csv")

# where to export the pLDDT-annotated PDBs shown below
EXPORT_DIR = Path("binder_structures_export")
EXPORT_DIR.mkdir(exist_ok=True)

res = pd.read_table(RESULTS_TSV)
old = pd.read_csv(OLD_CSV)

# match runs by TCR + pMHC sequence (targetids were renumbered)
key_cols = ["mhc", "peptide", "va", "ja", "cdr3a", "vb", "jb", "cdr3b"]
def seqkey(d):
    return d[key_cols].astype(str).agg("|".join, axis=1)
res["seqkey"] = seqkey(res)
old["seqkey"] = seqkey(old)

# bring 'query' (binder flag) and 'motif' (TCR group, e.g. TCR5) onto this run's rows
res = res.merge(old[["seqkey", "query", "motif"]].drop_duplicates("seqkey"),
                on="seqkey", how="left")
res = res.rename(columns={"motif": "tcr_group"})
print("rows:", len(res), "| binders (query==1):", int((res["query"] == 1).sum()))
res[["targetid", "tcr_group", "peptide", "pmhc_tcr_pae", "query"]].head()

rows: 102 | binders (query==1): 3


,targetid,tcr_group,peptide,pmhc_tcr_pae,query
0,T00000_A0201_CLAVEEVSL_0,TCR3,CLAVEEVSL,8.668093,0
1,T00001_A0201_KNCFRMTDQEAIQD_0,TCR34,KNCFRMTDQEAIQD,9.349597,0
2,T00002_A0201_CLAVEEVSL_0,TCR18,CLAVEEVSL,7.409732,0
3,T00003_A0201_KNCFRMTDQEAIQD_0,TCR3,KNCFRMTDQEAIQD,10.022885,0
4,T00004_A0201_CLAVEEVSL_0,TCR31,CLAVEEVSL,6.732878,0


## 2. Select binders and the highest-PAE structure per binder peptide

In [2]:
binders = res[res["query"] == 1].sort_values("pmhc_tcr_pae").reset_index(drop=True)

# For each binder, pick a high-PAE (worst-docking) NON-binder of the SAME peptide.
# Assign greedily by descending PAE and never reuse the same TCR group across
# partners, so every partner structure is a genuinely different TCR.
used_groups = set()
pairs = []  # (role, targetid, tcr_group, peptide, pae)
for _, b in binders.iterrows():
    pep = b["peptide"]
    cand = (res[(res["peptide"] == pep) & (res["query"] != 1)]
            .sort_values("pmhc_tcr_pae", ascending=False))
    # first candidate whose TCR group hasn't been used yet (fallback: worst overall)
    pick = cand[~cand["tcr_group"].isin(used_groups)]
    w = (pick.iloc[0] if len(pick) else cand.iloc[0])
    used_groups.add(w["tcr_group"])
    pairs.append(("binder", b["targetid"], b["tcr_group"], pep, b["pmhc_tcr_pae"]))
    pairs.append(("high-PAE same peptide", w["targetid"], w["tcr_group"], pep, w["pmhc_tcr_pae"]))

panel = pd.DataFrame(pairs, columns=["role", "targetid", "tcr_group", "peptide", "pmhc_tcr_pae"])
panel

,role,targetid,tcr_group,peptide,pmhc_tcr_pae
0,binder,T00074_A0201_AIQDLCLAV_0,TCR5,AIQDLCLAV,5.460901
1,high-PAE same peptide,T00084_A0201_AIQDLCLAV_0,TCR22,AIQDLCLAV,11.665605
2,binder,T00064_A0201_AIQDLCLAV_0,TCR29,AIQDLCLAV,5.549046
3,high-PAE same peptide,T00010_A0201_AIQDLCLAV_0,TCR10,AIQDLCLAV,11.468703
4,binder,T00073_A0201_CLAVEEVSL_0,TCR24,CLAVEEVSL,6.777761
5,high-PAE same peptide,T00099_A0201_CLAVEEVSL_0,TCR6,CLAVEEVSL,11.151324


## 3. Helpers: load PDB + apply pLDDT to B-factors

The PDB B-factors are 0, so we write the per-residue pLDDT (from the `.npy`) into the B-factor column, then color by it. AlphaFold pLDDT coloring convention:

- **blue** ≥ 90 (very high), **cyan** 70–90 (confident), **yellow** 50–70 (low), **orange** < 50 (very low).

In [3]:
def find_pdb(targetid):
    hits = list(PDB_DIR.glob(f"*{targetid}*model_2_ptm_ft4.pdb"))
    if not hits:
        raise FileNotFoundError(targetid)
    return hits[0]

def find_plddt(targetid):
    hits = list(NPY_DIR.glob(f"*{targetid}*_plddt.npy"))
    if not hits:
        raise FileNotFoundError(targetid)
    return hits[0]

# ---- MHC region used as the common reference frame ----
# chain layout (chainseq): MHC (175) / peptide / TCRa / TCRb. The MHC (A*02:01)
# is identical across all targets, so we superpose on its Cα atoms.
MHC_LEN = 175

def _residue_iter(pdb_path):
    """Yield (res_key, list_of_atom_line_indices) preserving order, plus lines."""
    lines = open(pdb_path).read().splitlines(keepends=True)
    order, groups, prev = [], {}, None
    for i, line in enumerate(lines):
        if line.startswith(("ATOM", "HETATM")):
            key = (line[21], line[22:27])
            if key != prev:
                order.append(key)
                groups[key] = []
                prev = key
            groups[key].append(i)
    return lines, order, groups

def _ca_coords(pdb_path, n_res):
    """Cα coordinates for the first n_res residues (the MHC), shape (n_res, 3)."""
    lines, order, groups = _residue_iter(pdb_path)
    coords = []
    for key in order[:n_res]:
        for i in groups[key]:
            if lines[i][12:16].strip() == "CA":
                coords.append([float(lines[i][30:38]), float(lines[i][38:46]),
                               float(lines[i][46:54])])
                break
    return np.asarray(coords)

def _kabsch(P, Q):
    """Rotation R and translation t mapping P onto Q (min RMSD)."""
    Pc, Qc = P.mean(0), Q.mean(0)
    H = (P - Pc).T @ (Q - Qc)
    U, S, Vt = np.linalg.svd(H)
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    D = np.diag([1, 1, d])
    R = Vt.T @ D @ U.T
    t = Qc - R @ Pc
    return R, t

# reference frame = MHC Cα of the lowest-PAE binder
REF_TID = binders.iloc[0]["targetid"]
REF_CA = _ca_coords(find_pdb(REF_TID), MHC_LEN)

def pdb_with_plddt(targetid, align=True):
    """PDB text with per-residue pLDDT in the B-factor column.
    If align=True, superpose onto the reference MHC frame (Kabsch on MHC Cα)."""
    plddt = np.load(find_plddt(targetid))
    lines, order, groups = _residue_iter(find_pdb(targetid))

    R = t = None
    if align:
        mob = _ca_coords(find_pdb(targetid), MHC_LEN)
        n = min(len(mob), len(REF_CA))
        R, t = _kabsch(mob[:n], REF_CA[:n])

    # residue index per line (for pLDDT lookup)
    ridx_of_line = {}
    for ri, key in enumerate(order):
        for i in groups[key]:
            ridx_of_line[i] = ri

    out_lines = []
    for i, line in enumerate(lines):
        if line.startswith(("ATOM", "HETATM")):
            ri = ridx_of_line[i]
            val = float(plddt[ri]) if ri < len(plddt) else 0.0
            x, y, z = float(line[30:38]), float(line[38:46]), float(line[46:54])
            if R is not None:
                x, y, z = R @ np.array([x, y, z]) + t
            out_lines.append(
                f"{line[:30]}{x:8.3f}{y:8.3f}{z:8.3f}{line[54:60]}{val:6.2f}{line[66:]}"
            )
        else:
            out_lines.append(line)
    return "".join(out_lines)

def save_pdb_with_plddt(targetid, role, tcr_group, peptide, pae):
    """Write the aligned, pLDDT-annotated PDB to EXPORT_DIR with an informative name."""
    role_tag = "binder" if role == "binder" else "highPAE"
    fname = f"{tcr_group}_{peptide}_{role_tag}_{targetid}_pae{pae:.2f}_plddt.pdb"
    dest = EXPORT_DIR / fname
    dest.write_text(pdb_with_plddt(targetid, align=True))
    return dest

print("reference frame (MHC Cα) from binder:", REF_TID, "| MHC residues:", len(REF_CA))

reference frame (MHC Cα) from binder: T00074_A0201_AIQDLCLAV_0 | MHC residues: 175


## 4. Visualise: 3 binders + the highest-PAE structure for each binder's peptide

Left column = binder (low PAE); right column = worst-docking model for the same peptide (high PAE). All colored by pLDDT.

In [4]:
from py3Dmol import view as _view

n_binders = len(binders)
# linked=True -> all viewers share the camera, so structures rotate/zoom together
grid = _view(viewergrid=(n_binders, 2), width=900, height=380 * n_binders,
             linked=True)
grid.setBackgroundColor("white")

# pLDDT gradient: low = red -> high = blue (red-white-blue).
# pLDDT clusters high (~70-98), so narrow the range to 50-90 for more contrast
# (regions below ~50 saturate red, making low-confidence areas obvious).
PLDDT_MIN, PLDDT_MAX = 50, 90
plddt_style = {"cartoon": {"colorscheme": {"prop": "b", "gradient": "rwb",
                                            "min": PLDDT_MIN, "max": PLDDT_MAX}}}

# panel rows are ordered binder, partner, binder, partner, ...
binder_rows = panel[panel["role"] == "binder"].reset_index(drop=True)
partner_rows = panel[panel["role"] != "binder"].reset_index(drop=True)

captions, saved = [], []
for row in range(n_binders):
    b = binder_rows.iloc[row]
    w = partner_rows.iloc[row]
    for col, (rec, role) in enumerate([(b, "binder"), (w, w["role"])]):
        tid, grp, pep, pae = rec["targetid"], rec["tcr_group"], rec["peptide"], rec["pmhc_tcr_pae"]
        pdb = pdb_with_plddt(tid)
        grid.addModel(pdb, "pdb", viewer=(row, col))
        grid.setStyle({}, plddt_style, viewer=(row, col))
        grid.zoomTo(viewer=(row, col))
        dest = save_pdb_with_plddt(tid, "binder" if col == 0 else "highPAE", grp, pep, pae)
        saved.append(dest)
        captions.append(f"[{row},{col}] {grp}  {role}: {tid}  peptide={pep}  PAE={pae:.2f}")

# panel labels are printed below (kept off the 3D canvas to avoid clutter)
print("\n".join(captions))
print(f"\nsaved {len(saved)} pLDDT-annotated PDBs to: {EXPORT_DIR.resolve()}")
print("viewers are linked: drag one structure and the others rotate/zoom with it")
for s in saved:
    print("  ", s.name)
grid.show()

[0,0] TCR5  binder: T00074_A0201_AIQDLCLAV_0  peptide=AIQDLCLAV  PAE=5.46
[0,1] TCR22  high-PAE same peptide: T00084_A0201_AIQDLCLAV_0  peptide=AIQDLCLAV  PAE=11.67
[1,0] TCR29  binder: T00064_A0201_AIQDLCLAV_0  peptide=AIQDLCLAV  PAE=5.55
[1,1] TCR10  high-PAE same peptide: T00010_A0201_AIQDLCLAV_0  peptide=AIQDLCLAV  PAE=11.47
[2,0] TCR24  binder: T00073_A0201_CLAVEEVSL_0  peptide=CLAVEEVSL  PAE=6.78
[2,1] TCR6  high-PAE same peptide: T00099_A0201_CLAVEEVSL_0  peptide=CLAVEEVSL  PAE=11.15

saved 6 pLDDT-annotated PDBs to: /data/yzy21/yy/af/tcrdock/cancer/tcrdock_cancer_viz/binder_structures_export
viewers are linked: drag one structure and the others rotate/zoom with it
   TCR5_AIQDLCLAV_binder_T00074_A0201_AIQDLCLAV_0_pae5.46_plddt.pdb
   TCR22_AIQDLCLAV_highPAE_T00084_A0201_AIQDLCLAV_0_pae11.67_plddt.pdb
   TCR29_AIQDLCLAV_binder_T00064_A0201_AIQDLCLAV_0_pae5.55_plddt.pdb
   TCR10_AIQDLCLAV_highPAE_T00010_A0201_AIQDLCLAV_0_pae11.47_plddt.pdb
   TCR24_CLAVEEVSL_binder_T00073_A0201_C

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [5]:
from IPython.display import HTML

# gradient matches 3Dmol 'rwb' over the narrowed pLDDT range (50 = red -> 90 = blue)
HTML(f"""
<div style="font-family:sans-serif;font-size:13px">
<b>pLDDT</b>
<div style="display:inline-block;width:260px;height:14px;vertical-align:middle;
     background:linear-gradient(to right,#ff0000,#ffffff,#0000ff);
     border:1px solid #999;margin:0 8px"></div>
<span>&le;{PLDDT_MIN} (low)</span> &nbsp;&rarr;&nbsp; <span>&ge;{PLDDT_MAX} (high)</span>
</div>
""")